# Phase 2: Data Cleaning and Preprocessing

## Objective

The objective of this phase is to improve the quality of the dataset before applying machine learning algorithms. This involves identifying and handling missing values, removing duplicate records, correcting inconsistent data formats, and converting features into a suitable format for analysis and model training.

A clean dataset improves the accuracy, reliability, and performance of machine learning models.

### Step 1: Import Required Libraries

The necessary Python libraries are imported to perform data cleaning and preprocessing operations. Pandas is used for handling the dataset, while NumPy is used for numerical computations.

In [889]:
import pandas as pd 
import numpy as np

### Step 2: Load the Dataset

The dataset is loaded into a Pandas DataFrame to begin the data cleaning process.

In [890]:
df = pd.read_csv("../dataset/Nepali_house_dataset.csv")

### Step 3: Create a Backup Copy

A backup copy of the original dataset is created to ensure that the raw data remains unchanged throughout the preprocessing phase.

In [891]:
df_clean=df.copy()

### Step 4: Identify Missing Values

Before cleaning the dataset, the number of missing values in each feature is examined. This helps determine the appropriate strategy for handling incomplete data.

In [892]:
df_clean.isnull().sum()

TITLE              0
LOCATION           0
PRICE              0
LAND AREA         89
BUILDUP AREA    2699
ROAD ACCESS        9
FACING           206
FLOOR             95
BEDROOM          282
BATHROOM         346
BUILT YEAR        61
PARKING         2786
AMENITIES          0
dtype: int64

In [893]:
missing=(df_clean.isnull().sum()/len(df_clean))*100
missing.sort_values(ascending=True)

TITLE            0.000000
LOCATION         0.000000
PRICE            0.000000
AMENITIES        0.000000
ROAD ACCESS      0.263312
BUILT YEAR       1.784669
LAND AREA        2.603862
FLOOR            2.779403
FACING           6.026916
BEDROOM          8.250439
BATHROOM        10.122879
BUILDUP AREA    78.964307
PARKING         81.509655
dtype: float64

### Step 6: Remove Duplicate Records

Duplicate records can affect model performance and introduce unnecessary bias. Therefore, duplicate entries are identified and removed from the dataset.

In [894]:
print("Before:", df_clean.shape)

df_clean.drop_duplicates(inplace=True)

print("After:", df_clean.shape)

Before: (3418, 13)
After: (3418, 13)


# Cleaning PRICE COLUMN

### Step 1: Analyze the PRICE Column

The **PRICE** column is the target variable of this project. Before training any machine learning model, it is essential to inspect the format of the price values. Since the values may contain different units such as Lakhs, Crores, or Rupees, we first examine the unique formats present in the dataset before converting them into a single numerical representation.

In [895]:
# Display first 20 prices
df_clean["PRICE"].head(20)

0      Rs. 2.9 Cr 
1     Rs. 4.75 Cr 
2     Rs. 1.99 Cr 
3        Rs. 4 Cr 
4     Rs. 12000000
5     Rs. 27000000
6      Rs. 3.3 Cr 
7        Rs. 4 Cr 
8      Rs. 4.5 Cr 
9         Rs. 3 Cr
10    Rs. 4.99 Cr 
11    Rs. 2.45 Cr 
12     Rs. 2.5 Cr 
13    Rs. 4.25 Cr 
14     Rs. 6.5 Cr 
15     Rs. 3.6 Cr 
16     Rs. 5.8 Cr 
17     Rs. 2.5 Cr 
18     Rs. 4.8 Cr 
19    Rs. 6.65 Cr 
Name: PRICE, dtype: str

In [896]:
# Display unique price formats
df_clean["PRICE"].unique()[:30]

<StringArray>
[ 'Rs. 2.9 Cr ', 'Rs. 4.75 Cr ', 'Rs. 1.99 Cr ',    'Rs. 4 Cr ',
 'Rs. 12000000', 'Rs. 27000000',  'Rs. 3.3 Cr ',  'Rs. 4.5 Cr ',
     'Rs. 3 Cr', 'Rs. 4.99 Cr ', 'Rs. 2.45 Cr ',  'Rs. 2.5 Cr ',
 'Rs. 4.25 Cr ',  'Rs. 6.5 Cr ',  'Rs. 3.6 Cr ',  'Rs. 5.8 Cr ',
  'Rs. 4.8 Cr ', 'Rs. 6.65 Cr ', 'Rs. 13.3 Cr ',  'Rs. 8.5 Cr ',
  'Rs. 5.5 Cr ', 'Rs. 2.95 Cr ', 'Rs. 6.35 Cr ', 'Rs. 10.5 Cr ',
  'Rs. 6.4 Cr ',  'Rs. 3.5 Cr ',  'Rs. 2.2 Cr ', 'Rs. 3.75 Cr ',
    'Rs. 8 Cr ', 'Rs. 7.75 Cr ']
Length: 30, dtype: str

In [897]:
# Check data type
df_clean["PRICE"].dtype

<StringDtype(storage='python', na_value=nan)>

In [898]:
# Number of unique price values
df_clean["PRICE"].nunique()

537

In [899]:
df_clean["PRICE"].sample(20, random_state=42)

1964      Rs. 3.45 Cr 
3187      Rs. 2.75 Cr 
170       Rs. 3.45 Cr 
680      Rs. 70,000 /m
2843       Rs. 1.6 Cr 
1763      Rs. 3.85 Cr 
3207       Rs.  3.8 Cr
1765      Rs. 3.35 Cr 
2827       Rs. 3.5 Cr 
2112      Rs. 3.78 Cr 
3237    Rs. 22,500,000
1108       Rs. 3.3 Cr 
194       Rs. 2.55 Cr 
1105      Rs. 2.25 Cr 
70        Rs. 3.25 Cr 
3374          Rs. 5 Cr
2321       Rs. 3.6 Cr 
1336        Rs. 21 Cr 
969       Rs. 1.55 Cr 
1621      Rs. 3.38 Cr 
Name: PRICE, dtype: str

### Step 2: Identify Rental Listings

The dataset is intended for predicting **house sale prices**. However, some records represent **monthly rental prices**, identified by the presence of `/m` in the `PRICE` column. These records are identified before being removed from the dataset to ensure consistency in the target variable.

In [900]:
# Find rows containing "/m"
rent_houses = df_clean[df_clean["PRICE"].str.contains("/m", na=False)]

rent_houses

,TITLE,LOCATION,PRICE,LAND AREA,BUILDUP AREA,ROAD ACCESS,FACING,FLOOR,BEDROOM,BATHROOM,BUILT YEAR,PARKING,AMENITIES
98,4 BHK House for Rent,"Bhaisepati, Lalitpur","Rs. 65,000 /m",4.0 aana,NaN,20 Feet,North-West,2.5,4.0,3.0,2070 B.S,1 CaRs. & 2 Bikes,"['Earthquake Resistant', 'Marbel', 'Parquet', ..."
109,5 BHK Bungalow for Rent,"Budhanilkantha, Kathmandu",Rs. 1.4 Lac/m,12.1 aana,NaN,16 Feet,East,2.5,5.0,5.0,2074 B.S,3 CaRs. & 3 Bikes,"['Marbel', 'Parquet', 'Earthquake Resistant', ..."
111,House for Rent,"Khumaltar, Lalitpur",Rs. 1.5 Lac/m,16.0 aana,NaN,20 Feet,NaN,2.0,6.0,3.0,2076 B.S,6 CaRs.,"['Marbel', 'Drainage', 'Garden', 'Parking', 'T..."
127,House for Rent,"Ranibari, Kathmandu",Rs. 1.05 Lac/m,5.1 aana,3119.28 Sq. Ft.,12 Feet,South-West,3.5,7.0,5.0,2074 B.S,2 CaRs.,"['Marbel', 'Drainage', 'Parking', 'Drinking Wa..."
159,House for Rent,"Naxal, Kathmandu",Rs. 2.5 Lac/m,24.0 aana,1096 Sq. Ft.,15 Feet,South,2.5,6.0,4.0,2073 B.S,10 CaRs. & 3 Bikes,"['Earthquake Resistant', 'Marbel', 'Parquet', ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3151,House for rent at Bhaisepati,"Bhaisepati, Lalitpur",Rs. 3 Lac/m,16.0 aana,NaN,0 Feet,South East,2.5,8.0,7.0,2071 B.S,NaN,"['Wifi', 'Drainage', 'Water Supply', 'Water Ta..."
3152,"House for rent at Sanepa, Lalitpur","Sanepa, Lalitpur",Rs. 1.75 Lac/m,8.0 aana,NaN,0 Feet,South,2.5,4.0,5.0,2076 B.S,NaN,['Parking']
3167,House for rent at Maharajgunj,"Maharajgunj, Kathmandu",Rs. 4.5 Lac/m,NaN,NaN,200 Feet,NaN,2.5,1.0,1.0,2070 B.S,NaN,"['Parquet', 'Marbel', 'Bathroom', 'Parking', '..."
3171,House on rent at Chappalkarkhana,"Chappal karkhana, Kathmandu",Rs. 3.3 Lac/m,24.0 aana,NaN,18 Feet,NaN,2.5,1.0,1.0,NaN,NaN,"['Bathroom', 'Drainage', 'Parking']"


In [901]:
# Find prices containing "Lac"
df_clean[df_clean["PRICE"].str.contains("Lac", case=False, na=False)]["PRICE"].unique()

<StringArray>
['Rs. 50 Lac/aana',   'Rs. 1.4 Lac/m',   'Rs. 1.5 Lac/m',  'Rs. 1.05 Lac/m',
 'Rs. 90 Lac/aana',   'Rs. 2.5 Lac/m', 'Rs. 70 Lac/aana', 'Rs. 21 Lac/aana',
   'Rs. 3.5 Lac/m',   'Rs. 1.6 Lac/m',
 ...
   'Rs. 2.68 Lac ',   'Rs. 98.5 Lac ',  'Rs. 1.99 Lac/m',  'Rs. 2.35 Lac/m',
 'Rs. 68 Lac/aana',     'Rs. 82 Lac ',   'Rs. 1.99 Lac ',   'Rs. 4.2 Lac/m',
 'Rs. 64 Lac/aana',    'Rs. 2.5 Lac ']
Length: 111, dtype: str

In [902]:
# Find prices containing "/"
df_clean[df_clean["PRICE"].str.contains("/", na=False)]["PRICE"].unique()

<StringArray>
['Rs. 50 Lac/aana',   'Rs. 65,000 /m',   'Rs. 1.4 Lac/m',   'Rs. 1.5 Lac/m',
  'Rs. 1.05 Lac/m', 'Rs. 90 Lac/aana',   'Rs. 2.5 Lac/m', 'Rs. 70 Lac/aana',
 'Rs. 21 Lac/aana',   'Rs. 60,000 /m',
 ...
  'Rs. 9.89 Lac/m',  'Rs. 1.49 Lac/m',   'Rs. 2.8 Lac/m',  'Rs. 2.55 Lac/m',
 'Rs. 62 Lac/aana',  'Rs. 1.99 Lac/m',  'Rs. 2.35 Lac/m', 'Rs. 68 Lac/aana',
   'Rs. 4.2 Lac/m', 'Rs. 64 Lac/aana']
Length: 108, dtype: str

In [903]:
# Find all non-Crore prices
df_clean[~df_clean["PRICE"].str.contains("Cr", na=False)]["PRICE"].unique()[:50]

<StringArray>
[   'Rs. 12000000',    'Rs. 27000000', 'Rs. 50 Lac/aana',   'Rs. 65,000 /m',
   'Price on call',   'Rs. 1.4 Lac/m',   'Rs. 1.5 Lac/m',  'Rs. 1.05 Lac/m',
 'Rs. 90 Lac/aana', 'Rs. 3,88,000,00',   'Rs. 2.5 Lac/m', 'Rs. 70 Lac/aana',
 'Rs. 21 Lac/aana',   'Rs. 60,000 /m',   'Rs. 3.5 Lac/m',   'Rs. 80,000 /m',
   'Rs. 70,000 /m',   'Rs. 45,000 /m',  'Rs. 27500000  ',   'Rs. 1.6 Lac/m',
  'Rs. 1.28 Lac/m',    'Rs. 1.5 Lac ',   'Rs. 90,000 /m',   'Rs. 1.7 Lac/m',
  'Rs. 1.85 Lac/m',     'Rs. 3 Lac/m',  'Rs. 1.55 Lac/m',     'Rs. 2 Lac/m',
   'Rs. 82,000 /m',   'Rs. 1.3 Lac/m',     'Rs. 1 Lac/m',   'Rs. 2.2 Lac/m',
     'Rs. 5 Lac/m',  'Rs. 1.75 Lac/m',   'Rs. 75,000 /m',     'Rs. 90 Lac ',
   'Rs. 1.1 Lac/m',     'Rs. 4 Lac/m',     'Rs. 6 Lac/m',  'Rs. 1.65 Lac/m',
     'Rs. 65 Lac ',  'Rs. 2.25 Lac/m',  'Rs. 3.35 Lac/m',   'Rs. 4.5 Lac/m',
  'Rs. 1.25 Lac/m',  'Rs. 12.7 Lac/m',  'Rs. 2.75 Lac/m',   'Rs. 85,000 /m',
     'Rs. 15 Lac ',  'Price on call ']
Length: 50, dtype: str

### Step 3: Remove Invalid Price Records

The dataset contains different types of price representations, including monthly rent, price per Aana, price per square foot, and listings with unavailable prices ("Price on call"). Since the objective of this project is to predict the total selling price of houses, these records are removed to maintain consistency in the target variable.

In [904]:
invalid_patterns = [
    "/m",
    "/aana",
    "/sf",
    "Price on call"
]

mask = ~df_clean["PRICE"].str.contains(
    "|".join(invalid_patterns),
    case=False,
    na=False
)

df_clean = df_clean[mask]
df_clean.shape

(2667, 13)

### Step 4: Convert Price into Numerical Format

The values in the **PRICE** column are stored in different textual formats such as Crores and Rupees with commas. Machine learning algorithms require numerical values, so all prices are converted into a single numerical representation in Nepalese Rupees.

In [905]:
import re
def clean_price(price):
    price = str(price).strip()

    #Remove "Rs."
    price=price.replace("Rs.","").strip()

    #Remove commas
    price=price.replace(",","")

    #convert crores to rupess
    if "Cr" in price:
        price = price.replace("Cr","").strip()
        return float(price)*10000000

     # Lakhs
    elif "Lac" in price:
        value = float(price.replace("Lac", "").strip())
        return value * 100000

    # Already numeric
    else:
        return float(price)

df_clean["PRICE"]=df_clean["PRICE"].apply(clean_price)

In [906]:
df_clean["PRICE"].head(50)

0      29000000.0
1      47500000.0
2      19900000.0
3      40000000.0
4      12000000.0
5      27000000.0
6      33000000.0
7      40000000.0
8      45000000.0
9      30000000.0
10     49900000.0
11     24500000.0
12     25000000.0
13     42500000.0
14     65000000.0
15     36000000.0
16     58000000.0
17     25000000.0
18     48000000.0
19     66500000.0
20    133000000.0
21     85000000.0
22     55000000.0
23     47500000.0
24     29500000.0
25     63500000.0
26     85000000.0
27    105000000.0
28     64000000.0
29     35000000.0
30     22000000.0
31     37500000.0
32     80000000.0
33     55000000.0
34     55000000.0
35     77500000.0
36    160000000.0
37     31900000.0
38     25000000.0
39    120000000.0
40     62500000.0
41    160000000.0
42     28000000.0
43     59900000.0
44     48000000.0
45     25000000.0
46     33500000.0
47     35000000.0
48     40000000.0
49     60000000.0
Name: PRICE, dtype: float64

# Cleaning LAND AREA COLUMN

### Step 1: Analyze the LAND AREA Column

The **LAND AREA** column represents the total land size of the property. The values are stored using different measurement units such as Aana, Kattha, Dhur, Ropani, Square Feet, and Square Meter. Before applying machine learning algorithms, all land area values must be converted into a single standard unit.

In [907]:
df_clean["LAND AREA"].sample(50, random_state=42)

354        3.2 aana
2384       5.3 aana
2111       4.0 aana
3198       4.0 aana
1223       3.3 aana
673        3.0 aana
775        3.2 aana
659        3.0 aana
2057       4.2 aana
1587       4.0 aana
3034       5.0 aana
2786       4.0 aana
3311         3 aana
2828       5.1 aana
343     0.12 kattha
1681       6.0 aana
830        3.3 aana
2831       4.0 aana
2938       3.2 aana
2376       6.0 aana
1769       3.0 aana
3226       4.1 aana
2203       8.0 aana
584        4.1 aana
826        3.1 aana
825        4.2 aana
1602       3.2 aana
1332       5.0 aana
3264       4.2 aana
1484         3 aana
1937       4.3 aana
536        2.2 aana
3286        4 aana 
32         6.0 aana
774        3.0 aana
3196      12.0 aana
1489       8.0 aana
2671       6.0 aana
2455       4.0 aana
3387       4.7 aana
627        2.3 aana
2210      10.0 aana
73         2.3 aana
521        2.5 aana
70         4.3 aana
1447       3.0 aana
725        2.3 aana
1703      10.0 aana
2025       9.2 aana
1416       3.2 aana


In [908]:
df_clean["LAND AREA"].dropna().unique()[:100]

<StringArray>
[    '4.0 aana',     '3.0 aana',     '2.3 aana',     '7.0 aana',
     '6.0 aana',     '3.2 aana',     '4.3 aana',     '2.2 aana',
     '5.0 aana',     '5.2 aana',     '6.2 aana',     '4.2 aana',
     '4.1 aana',     '9.6 aana',     '3.1 aana',    '11.0 aana',
     '3.3 aana',     '9.0 aana',      '12 aana',     '6.4 aana',
    '12.0 aana',     '3.5 aana',       '6 aana',     '7.1 aana',
     '6.3 aana',     '9.3 aana',     '9.1 aana',     '6.1 aana',
     '2.9 aana',     '7.2 aana',    '10.2 aana',     '8.0 aana',
    '15.0 aana',     '7.3 aana',     '5.1 aana',    '10.0 aana',
     '5.3 aana',     '2.5 aana',  '0.12 kattha',   '0.5 kattha',
    '14.0 aana',     '1.2 aana',     '8.2 aana',     '1.1 aana',
     '3.8 aana',    '13.0 aana',    '16.0 aana', '1.9.9 kattha',
  '0.11 kattha',     '9.2 aana',     '3.4 aana',     '2.8 aana',
    '17.0 aana',   '1.7 kattha',   '1.3 kattha',       '3 aana',
    '11.2 aana',    '10.3 aana',  '0.15 kattha',  '0.14 kattha',
   '3.2 kat

In [909]:
# Count different units
print(df_clean["LAND AREA"].str.contains("aana", case=False, na=False).sum())

print(df_clean["LAND AREA"].str.contains("kattha", case=False, na=False).sum())

print(df_clean["LAND AREA"].str.contains("sq", case=False, na=False).sum())

2609
27
11


In [910]:
df_clean[
    df_clean["LAND AREA"].str.contains(r"\.\d+\.\d+", regex=True, na=False)
]["LAND AREA"]

463    1.9.9 kattha
Name: LAND AREA, dtype: str

### Step 7: Standardize the LAND AREA Feature

The **LAND AREA** feature contains measurements in multiple units such as **Aana**, **Kattha**, and **Square Feet**. Machine learning models require numerical values in a consistent unit. Therefore, all land area measurements are converted into **Square Feet (sq.ft)** using standard Nepali land conversion factors.

Conversion factors used:

- 1 Aana = 342.25 sq.ft
- 1 Kattha = 3645 sq.ft
- 1 Square Foot = 1 sq.ft

In [911]:
import re

def convert_land_area(area):
    if pd.isna(area):
        return np.nan

    area = str(area).strip().lower()

    # Remove extra spaces
    area = re.sub(r"\s+", " ", area)

    # Fix known typo
    area = area.replace("1.9.9", "1.99")

    # Invalid value
    if area == "0 sq. ft":
        return np.nan

    # Aana → Square Feet
    if "aana" in area:
        value = float(area.replace("aana", "").strip())
        return value * 342.25

    # Kattha → Square Feet
    elif "kattha" in area:
        value = float(area.replace("kattha", "").strip())
        return value * 3645

    # Square Feet
    elif "sq. ft" in area:
        value = float(area.replace("sq. ft", "").strip())
        return value

    return np.nan
df_clean["LAND AREA"] = df_clean["LAND AREA"].apply(convert_land_area)

In [912]:
df_clean[["LAND AREA"]].head(20)

,LAND AREA
0,1369.000
1,1026.750
2,787.175
3,2395.750
4,2053.500
5,2053.500
6,1095.200
7,1471.675
8,752.950
9,1711.250


In [913]:
df_clean["LAND AREA"].dtype

dtype('float64')

In [914]:
df_clean["LAND AREA"].isnull().sum()

np.int64(21)

In [915]:
df_clean["LAND AREA"].describe()

count     2646.000000
mean      1730.681679
std       1403.189769
min          4.500000
25%       1095.200000
50%       1369.000000
75%       1779.700000
max      36450.000000
Name: LAND AREA, dtype: float64

In [916]:
df_clean[df_clean["LAND AREA"] < 100]

,TITLE,LOCATION,PRICE,LAND AREA,BUILDUP AREA,ROAD ACCESS,FACING,FLOOR,BEDROOM,BATHROOM,BUILT YEAR,PARKING,AMENITIES
1783,"Commercial building for sale in Machhapokhari,...","Machhapokhari, Kathmandu",55000000.0,4.5,9500 Sq. Feet,20 Feet,South-East,5.5,12.0,6.0,2076 B.S,NaN,"['Drainage', 'Drinking Water', 'Power Backup',..."


In [917]:
# Remove unrealistic land area values
df_clean = df_clean[df_clean["LAND AREA"] >= 100]

In [918]:
df_clean["LAND AREA"].describe()

count     2645.00000
mean      1731.33430
std       1403.05339
min        102.67500
25%       1095.20000
50%       1369.00000
75%       1779.70000
max      36450.00000
Name: LAND AREA, dtype: float64

# Inspect Remaining Columns

In [919]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 2645 entries, 0 to 3417
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   TITLE         2645 non-null   str    
 1   LOCATION      2645 non-null   str    
 2   PRICE         2645 non-null   float64
 3   LAND AREA     2645 non-null   float64
 4   BUILDUP AREA  562 non-null    str    
 5   ROAD ACCESS   2643 non-null   str    
 6   FACING        2586 non-null   str    
 7   FLOOR         2606 non-null   float64
 8   BEDROOM       2469 non-null   float64
 9   BATHROOM      2413 non-null   float64
 10  BUILT YEAR    2622 non-null   str    
 11  PARKING       542 non-null    str    
 12  AMENITIES     2645 non-null   str    
dtypes: float64(5), str(8)
memory usage: 289.3 KB


In [920]:
missing = pd.DataFrame({
    "Missing Count": df_clean.isnull().sum(),
    "Missing %": round(df_clean.isnull().sum() / len(df_clean) * 100, 2)
})

missing.sort_values("Missing %", ascending=False)

,Missing Count,Missing %
PARKING,2103,79.51
BUILDUP AREA,2083,78.75
BATHROOM,232,8.77
BEDROOM,176,6.65
FACING,59,2.23
FLOOR,39,1.47
BUILT YEAR,23,0.87
ROAD ACCESS,2,0.08
TITLE,0,0.00
LAND AREA,0,0.00


# Clean BUILDUP Area

The **BUILDUP AREA** column contained approximately 80% missing values. Since such a large proportion of missing data would make imputation unreliable and could introduce bias into the model, the feature was removed from the dataset.

In [921]:
df_clean.drop(columns=["BUILDUP AREA"], inplace=True)

# Clean ROAD ACCESS

In [922]:
df_clean["ROAD ACCESS"].dropna().unique()[:50]

<StringArray>
[   '12 Feet',    '10 Feet',    '20 Feet',    '13 Feet',    '14 Feet',
    '16 Feet',    '26 Feet',    '25 Feet',    '19 Feet',    '18 Feet',
    '15 Feet',    '24 Feet',    '27 Feet',    '30 Feet',    '23 Feet',
     '8 Feet',    '11 Feet',    '22 Feet',   '20 Meter',     '6 Feet',
     '4 Feet',   '10 Meter',    '17 Feet',   '13 Meter', '13-20 Feet',
 '12-18 Feet', '10-12 Feet', '10-15 Feet', '12-16 Feet',    '32 Feet',
 '12/20 Feet',     '9 Feet', '12-14 Feet', '15-26 Feet', '10-20 Feet',
 '14-20 Feet', '12-13 Feet',  '8-10 Feet', '12/13 Feet',   '20  Feet',
  '9-12 Feet',   '13  Feet', '10/13 Feet', '10-24 Feet', '16-22 Feet',
 '20-26 Feet', '13-16 Feet', '12-15 Feet',  '8-12 Feet', '15-24 Feet']
Length: 50, dtype: str

In [923]:
import re

def clean_road(value):

    # Missing value
    if pd.isna(value):
        return np.nan

    # Convert everything to string
    value = str(value).lower().strip()

    # Replace "/" with "-"
    value = value.replace("/", "-")

    # Extract all numbers
    numbers = re.findall(r"\d+\.?\d*", value)

    if len(numbers) == 0:
        return np.nan

    numbers = [float(x) for x in numbers]

    # Average if there are multiple numbers
    road = sum(numbers) / len(numbers)

    # Convert meters to feet
    if "meter" in value:
        road *= 3.28084

    return road

df_clean["ROAD ACCESS"] = df_clean["ROAD ACCESS"].apply(clean_road)

In [924]:
df_clean["ROAD ACCESS"].head(20)

0     12.0
1     10.0
2     10.0
3     12.0
4     20.0
5     12.0
6     13.0
7     14.0
8     10.0
9     12.0
10    13.0
11    16.0
12    13.0
13    20.0
14    12.0
15    13.0
16    14.0
17    12.0
18    16.0
19    20.0
Name: ROAD ACCESS, dtype: float64

In [925]:
df_clean["ROAD ACCESS"].describe()

count    2643.000000
mean       14.954061
std         4.770544
min         0.000000
25%        13.000000
50%        13.000000
75%        16.000000
max        82.000000
Name: ROAD ACCESS, dtype: float64

### Handling Invalid Road Access Values

During data inspection, several properties were found with a road access value of **0 feet**. Since a road width of zero is not physically meaningful, these values were treated as missing (`NaN`) rather than valid measurements. The missing values were then imputed using the median road width to preserve the records while maintaining data quality.

In [926]:
df_clean[df_clean["ROAD ACCESS"] < 5]

,TITLE,LOCATION,PRICE,LAND AREA,ROAD ACCESS,FACING,FLOOR,BEDROOM,BATHROOM,BUILT YEAR,PARKING,AMENITIES
375,4 BHK House for Sale,"Nakhudole, Lalitpur",15500000.0,1437.450,4.0,South,2.0,4.0,2.0,2072 B.S,1 CaRs. & 1 Bikes,"['Earthquake Resistant', 'Parking', 'Power Bac..."
736,"House on sale at Satungal, Chandragiri","Satungal, Kathmandu",15000000.0,752.950,4.0,East,3.0,3.0,3.0,2072 B.S,NaN,"['Parking', 'Drinking Water', 'Bathroom']"
753,"House for sale at Pabitranagar, New Buspark","New Buspark, Kathmandu",40000000.0,2053.500,4.0,West,2.5,6.0,4.0,2073 B.S,NaN,"['Marbel', 'Drinking Water', 'Bathroom', 'Drai..."
792,Beautiful House for sale in Dhapakhel,"Dhapakhel, Lalitpur",27500000.0,3080.250,4.0,North-East,NaN,NaN,NaN,2072 B.S,NaN,"['Marbel', 'Bathroom', 'Drinking Water', 'Park..."
1185,"House on sale in land price near Maya Nursery,...","Kalopool, Kathmandu",30000000.0,1437.450,4.0,South,1.0,2.0,2.0,2070 B.S,NaN,"['Drainage', 'Bathroom', 'Drinking Water', 'Pa..."
1362,House for sale at Ramkot,"Sitapaila, Kathmandu",12000000.0,787.175,4.0,North-East,2.5,3.0,NaN,2079 B.S,NaN,"['Washing Machine', 'Parking', 'Drinking Water..."
1534,"Residential house for sale in Saibu, Bhaisepati","Sainbu, Lalitpur",30000000.0,1369.000,0.0,North,2.5,4.0,NaN,2073 B.S,NaN,"['Drinking Water', 'Bathroom', 'Drainage', 'Pa..."
1640,"House on sale near Madan Bhandari College, Ana...","Anamnagar, Kathmandu",13000000.0,1026.750,4.0,West,4.0,4.0,2.0,2071 B.S,NaN,"['Drainage', 'Bathroom', 'Drinking Water']"
2983,Bungalow on sale in Budhanilkantha,"Rudreshwor, Kathmandu",140000000.0,4107.000,0.0,North,NaN,5.0,NaN,2076 B.S,NaN,"['Drinking Water', 'Bathroom', 'Drainage', 'Ga..."
3029,House on sale at Kapan Chunikhel,"Chunikhel, Kathmandu",23000000.0,1095.200,0.0,West,2.5,6.0,4.0,2079 B.S,NaN,"['Parking', 'Drainage', 'Water Supply', 'Power..."


In [927]:
df_clean["ROAD ACCESS"] = df_clean["ROAD ACCESS"].replace(0, np.nan)

In [928]:
df_clean["ROAD ACCESS"].describe()

count    2636.000000
mean       14.993772
std         4.714118
min         4.000000
25%        13.000000
50%        13.000000
75%        16.000000
max        82.000000
Name: ROAD ACCESS, dtype: float64

In [929]:
df_clean["ROAD ACCESS"].fillna(
    df_clean["ROAD ACCESS"].median(),
    
)

0       12.0
1       10.0
2       10.0
3       12.0
4       20.0
        ... 
3413    16.0
3414    16.0
3415    16.0
3416    16.0
3417    16.0
Name: ROAD ACCESS, Length: 2645, dtype: float64

# TITLE Column

The **TITLE** column was analyzed to identify the type of property listed.

The project focuses on predicting **residential house prices**, therefore only residential property listings were retained.

The following property types were kept:

- House
- Bungalow
- Villa

The following property types were removed:

- Commercial Buildings
- Land
- Flats
- Apartments

These property categories have different pricing characteristics and would introduce unnecessary variation into the machine learning model.

After filtering, the **TITLE** column was dropped because it is an unstructured text feature and was not used for model training.

In [930]:
df_clean[df_clean["TITLE"].str.contains("rent", case=False, na=False)]

,TITLE,LOCATION,PRICE,LAND AREA,ROAD ACCESS,FACING,FLOOR,BEDROOM,BATHROOM,BUILT YEAR,PARKING,AMENITIES
4,House for Rent,"Maharajgunj, Kathmandu",12000000.0,2053.500,20.0,South,2.0,4.0,4.0,2071 B.S,4 CaRs. & 5 Bikes,"['Earthquake Resistant', 'Parquet', 'Parking',..."
5,Bungalow House for Rent,"Bhaisepati, Lalitpur",27000000.0,2053.500,12.0,South-East,2.5,6.0,5.0,2076 B.S,4 CaRs. & 5 Bikes,"['Earthquake Resistant', 'Marbel', 'Parquet', ..."
46,4 BHK House for Rent,"Sanepa, Lalitpur",33500000.0,1711.250,19.0,West,2.5,4.0,2.0,2076 B.S,1 CaRs. & 2 Bikes,"['Earthquake Resistant', 'Drinking Water', 'Ma..."
47,4 BHK House for Rent,"Sanepa, Lalitpur",35000000.0,1711.250,19.0,North-West,2.5,4.0,3.0,2075 B.S,1 CaRs. & 2 Bikes,"['Marbel', 'Parquet', 'Earthquake Resistant', ..."
87,Bungalow House for Rent,"Baluwatar, Kathmandu",22500000.0,1369.000,20.0,South-East,2.5,4.0,4.0,2068 B.S,5 CaRs. & 5 Bikes,"['Earthquake Resistant', 'Marbel', 'Parquet', ..."
116,House for Rent,"Muhanpokhari, Kathmandu",20000000.0,1095.200,20.0,South,3.0,7.0,5.0,2074 B.S,4 CaRs. & 5 Bikes,[]
140,House for Rent,"Dhapakhel, Lalitpur",38800000.0,1369.000,10.0,East,5.0,5.0,7.0,2072 B.S,4 Bikes,"['Marbel', 'Parquet', 'Drainage', 'Power Backu..."
157,4 BHK House for Rent,"Bhaisepati, Lalitpur",28500000.0,1369.000,20.0,North-West,2.5,4.0,3.0,2076 B.S,1 CaRs. & 2 Bikes,"['Marbel', 'Earthquake Resistant', 'Power Back..."
176,House for Rent,"Satdobato, Lalitpur",15000000.0,2053.500,15.0,South,2.0,4.0,4.0,2064 B.S,4 CaRs. & 4 Bikes,"['Power Backup', 'Marbel', 'Earthquake Resista..."
179,House for Rent,"Basundhara, Kathmandu",27500000.0,1095.200,20.0,South,2.5,7.0,6.0,2076 B.S,4 CaRs. & 5 Bikes,"['Earthquake Resistant', 'Parquet', 'Drainage'..."


In [931]:
df_clean["TITLE"].str.contains("rent", case=False, na=False).sum()

np.int64(45)

In [932]:
df_clean = df_clean[
    ~df_clean["TITLE"].str.contains("rent", case=False, na=False)
]

In [933]:
df_clean[df_clean["TITLE"].str.contains("rent", case=False, na=False)]

,TITLE,LOCATION,PRICE,LAND AREA,ROAD ACCESS,FACING,FLOOR,BEDROOM,BATHROOM,BUILT YEAR,PARKING,AMENITIES


In [934]:
df_clean["PROPERTY_TYPE"] = (
    df_clean["TITLE"]
    .str.lower()
    .str.extract(r"(house|bungalow|commercial|villa|flat|land|apartment)")
)

In [935]:
df_clean["PROPERTY_TYPE"].value_counts(dropna=False)

PROPERTY_TYPE
house         2072
bungalow       298
commercial     103
NaN             83
villa           22
flat            15
land             6
apartment        1
Name: count, dtype: int64

In [936]:
before = len(df_clean)

allowed = ["house", "bungalow", "villa"]
df_clean = df_clean[df_clean["PROPERTY_TYPE"].isin(allowed)]

after = len(df_clean)

print(f"Rows before filtering: {before}")
print(f"Rows after filtering : {after}")
print(f"Rows removed          : {before - after}")

Rows before filtering: 2600
Rows after filtering : 2392
Rows removed          : 208


# Floor Cleaning

The **FLOOR** column was already stored as a numeric variable (`float64`), requiring no type conversion.

The column contained **38 missing values**, representing approximately **1.6%** of the dataset. Since the missing proportion was very small and the distribution was centered around **2.5 floors**, missing values were imputed using the **median**.

No unrealistic floor values were identified, and therefore all remaining records were retained.

In [937]:
print(df_clean["FLOOR"].dtype)

print(df_clean["FLOOR"].unique()[:100])

print(df_clean["FLOOR"].isnull().sum())

print(df_clean["FLOOR"].describe())

float64
[3.  4.5 2.5 5.5 4.  3.5 2.  1.  1.5 5.  nan 2.4 6.5]
38
count    2354.000000
mean        2.595752
std         0.574895
min         1.000000
25%         2.500000
50%         2.500000
75%         2.500000
max         6.500000
Name: FLOOR, dtype: float64


In [938]:
df_clean[df_clean["FLOOR"] == 2.4]

,TITLE,LOCATION,PRICE,LAND AREA,ROAD ACCESS,FACING,FLOOR,BEDROOM,BATHROOM,BUILT YEAR,PARKING,AMENITIES,PROPERTY_TYPE
900,"New house for sale at Kapan, Budhanilkantha","Kapan, Kathmandu",29500000.0,1369.0,13.0,West,2.4,6.0,5.0,2074 B.S,NaN,"['Drinking Water', 'Bathroom', 'Drainage', 'Pa...",house


In [939]:
df_clean["FLOOR"] = df_clean["FLOOR"].fillna(
    df_clean["FLOOR"].median()
)

In [940]:
print(df_clean["FLOOR"].isnull().sum())

print(df_clean["FLOOR"].describe())

0
count    2392.000000
mean        2.594231
std         0.570434
min         1.000000
25%         2.500000
50%         2.500000
75%         2.500000
max         6.500000
Name: FLOOR, dtype: float64


# Cleaning Bedroom

The **BEDROOM** column was already stored as a numeric (`float64`) feature and required no type conversion.

The dataset contained **148 missing values**, which were imputed using the **median bedroom count (5)**.

The column was also examined for unusually high bedroom counts using the Interquartile Range (IQR) method. Although a few properties had exceptionally high numbers of bedrooms, these records were retained because they may represent genuine large residential buildings rather than data-entry errors. Removing valid observations could reduce the model's ability to generalize to larger properties.

In [941]:
print("Data Type:")
print(df_clean["BEDROOM"].dtype)

print("\nUnique Values:")
print(sorted(df_clean["BEDROOM"].dropna().unique()))

print("\nMissing Values:")
print(df_clean["BEDROOM"].isnull().sum())

print("\nStatistics:")
print(df_clean["BEDROOM"].describe())

Data Type:
float64

Unique Values:
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(22.0), np.float64(28.0), np.float64(36.0)]

Missing Values:
148

Statistics:
count    2244.000000
mean        5.193405
std         1.940265
min         1.000000
25%         4.000000
50%         5.000000
75%         6.000000
max        36.000000
Name: BEDROOM, dtype: float64


In [942]:
df_clean["BEDROOM"].value_counts().sort_index()

BEDROOM
1.0       8
2.0      77
3.0     132
4.0     609
5.0     616
6.0     470
7.0     172
8.0      84
9.0      21
10.0     17
11.0     13
12.0     10
13.0      4
14.0      4
16.0      1
17.0      2
18.0      1
22.0      1
28.0      1
36.0      1
Name: count, dtype: int64

In [943]:
df_clean[df_clean["BEDROOM"] >= 15][
    ["LOCATION", "PRICE", "LAND AREA", "FLOOR", "BEDROOM", "BATHROOM"]
]


,LOCATION,PRICE,LAND AREA,FLOOR,BEDROOM,BATHROOM
323,"Old Baneshwor, Kathmandu",29000000.0,1095.20,4.5,22.0,8.0
363,"Dillibazar, Kathmandu",140000000.0,4791.50,4.5,28.0,7.0
1155,"Lainchaur, Kathmandu",43000000.0,1711.25,3.5,17.0,10.0
1389,"Balkumari, Lalitpur",42500000.0,1437.45,4.0,16.0,4.0
1789,"Shantinagar, Kathmandu",52500000.0,2121.95,4.0,36.0,17.0
1943,"Guheshwori, Kathmandu",35000000.0,1369.00,4.5,18.0,4.0
2074,"Manbhawan, Lalitpur",62000000.0,2464.20,5.0,17.0,10.0


In [944]:
Q1 = df_clean["BEDROOM"].quantile(0.25)
Q3 = df_clean["BEDROOM"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(lower)
print(upper)

1.0
9.0


# Bathroom

In [945]:
print("Data Type:")
print(df_clean["BATHROOM"].dtype)

print("\nUnique Values:")
print(sorted(df_clean["BATHROOM"].dropna().unique()))

print("\nMissing Values:")
print(df_clean["BATHROOM"].isnull().sum())

print("\nStatistics:")
print(df_clean["BATHROOM"].describe())

print("\nValue Counts:")
print(df_clean["BATHROOM"].value_counts().sort_index())

Data Type:
float64

Unique Values:
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(12.0), np.float64(17.0), np.float64(32.0)]

Missing Values:
204

Statistics:
count    2188.000000
mean        3.981261
std         1.465380
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max        32.000000
Name: BATHROOM, dtype: float64

Value Counts:
BATHROOM
1.0      91
2.0     114
3.0     539
4.0     796
5.0     443
6.0     134
7.0      44
8.0      17
9.0       3
10.0      4
12.0      1
17.0      1
32.0      1
Name: count, dtype: int64


In [946]:
df_clean[df_clean["BATHROOM"] >= 10][
    ["LOCATION",
     "PRICE",
     "LAND AREA",
     "FLOOR",
     "BEDROOM",
     "BATHROOM"]
]

,LOCATION,PRICE,LAND AREA,FLOOR,BEDROOM,BATHROOM
344,"Baluwatar, Kathmandu",76000000.0,2053.500,3.5,8.0,10.0
1155,"Lainchaur, Kathmandu",43000000.0,1711.250,3.5,17.0,10.0
1377,"Simaltar, Kathmandu",95000000.0,4791.500,3.0,9.0,10.0
1683,"Deuwa chowk, Kathmandu",70000000.0,2429.975,4.0,8.0,12.0
1789,"Shantinagar, Kathmandu",52500000.0,2121.950,4.0,36.0,17.0
1956,"Kapan, Kathmandu",33000000.0,1129.425,2.5,6.0,32.0
2074,"Manbhawan, Lalitpur",62000000.0,2464.200,5.0,17.0,10.0


In [947]:
df_clean["BATHROOM"] = df_clean["BATHROOM"].fillna(
    df_clean["BATHROOM"].median()
)

In [948]:
print(df_clean["BATHROOM"].isnull().sum())

print(df_clean["BATHROOM"].describe())

0
count    2392.000000
mean        3.982860
std         1.401484
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max        32.000000
Name: BATHROOM, dtype: float64


# Cleaning Built Year

The **BUILT YEAR** column contained construction years stored as text in Bikram Sambat (B.S.) format (e.g., `2076 B.S`). A few records contained only the four-digit year (e.g., `2075`) without the `B.S.` suffix.

A regular expression was used to extract the four-digit year from each value, and the column was converted into a numeric format.

The dataset contained **22 missing values** (less than 1% of the records). These missing values were imputed using the **median built year**, ensuring that the overall distribution of construction years was preserved.

The cleaned construction year was retained for modeling. In a later phase, an additional feature representing the **age of the house** can be derived from this column.

In [949]:
print("Data Type:")
print(df_clean["BUILT YEAR"].dtype)

print("\nUnique Values:")
print(df_clean["BUILT YEAR"].dropna().unique()[:50])

print("\nMissing Values:")
print(df_clean["BUILT YEAR"].isnull().sum())

print("\nValue Counts:")
print(df_clean["BUILT YEAR"].value_counts().head(20))

Data Type:
str

Unique Values:
<StringArray>
['2076 B.S', '2060 B.S', '2059 B.S', '2074 B.S', '2065 B.S', '2066 B.S',
 '2075 B.S', '2079 B.S', '2070 B.S', '2078 B.S', '2077 B.S', '2080 B.S',
 '2071 B.S', '2068 B.S', '2073 B.S', '2072 B.S', '2063 B.S', '2064 B.S',
 '2069 B.S', '2055 B.S', '2061 B.S', '2058 B.S', '2050 B.S', '2062 B.S',
 '2049 B.S', '2047 B.S', '2054 B.S', '2056 B.S', '2067 B.S', '2052 B.S',
 '2057 B.S',     '2060',     '2065',     '2073',     '2075',     '2071']
Length: 36, dtype: str

Missing Values:
22

Value Counts:
BUILT YEAR
2076 B.S    368
2078 B.S    271
2074 B.S    201
2070 B.S    165
2075 B.S    163
2072 B.S    150
2077 B.S    148
2079 B.S    137
2073 B.S    106
2068 B.S     94
2065 B.S     77
2060 B.S     75
2069 B.S     70
2071 B.S     58
2062 B.S     47
2058 B.S     35
2066 B.S     25
2080 B.S     24
2063 B.S     21
2064 B.S     20
Name: count, dtype: int64


In [950]:
import re
import numpy as np

def clean_built_year(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # Extract the first 4-digit number
    match = re.search(r"\d{4}", value)

    if match:
        return int(match.group())

    return np.nan

df_clean["BUILT YEAR"] = df_clean["BUILT YEAR"].apply(clean_built_year)

In [951]:
df_clean["BUILT YEAR"] = df_clean["BUILT YEAR"].fillna(
    df_clean["BUILT YEAR"].median()
)

In [952]:
print(df_clean["BUILT YEAR"].dtype)

print(df_clean["BUILT YEAR"].head())

print(df_clean["BUILT YEAR"].describe())

print(df_clean["BUILT YEAR"].isnull().sum())

float64
0    2076.0
1    2076.0
2    2060.0
3    2059.0
6    2074.0
Name: BUILT YEAR, dtype: float64
count    2392.000000
mean     2072.224916
std         6.165805
min      2047.000000
25%      2070.000000
50%      2074.000000
75%      2076.000000
max      2080.000000
Name: BUILT YEAR, dtype: float64
0


In [953]:
CURRENT_BS_YEAR = 2083

df_clean["HOUSE AGE"] = CURRENT_BS_YEAR - df_clean["BUILT YEAR"]

# Cleaning Facing

In [954]:
print("Data Type:")
print(df_clean["FACING"].dtype)

print("\nUnique Values:")
print(df_clean["FACING"].dropna().unique())

print("\nMissing Values:")
print(df_clean["FACING"].isnull().sum())

print("\nValue Counts:")
print(df_clean["FACING"].value_counts())

Data Type:
str

Unique Values:
<StringArray>
[        'West',   'North-West',   'North-East',         'East',
        'South',   'South-East',   'South-West',        'North',
   'South East',        'south',   'South West',   'North East',
   'North West',         'west',         'EAST',   'West-South',
         'WEST',  'North- East',   'South-EAST',        'SOUTH',
   'East/South',   'NORTH-WEST',        'NORTH',   'SOUTH-EAST',
   'EAST-SOUTH',   'SOUTH-WEST', 'WEST / NORTH',   'EAST-NORTH',
   'East-North',   'NORTH/EAST',   'South-east',   'East-South']
Length: 32, dtype: str

Missing Values:
52

Value Counts:
FACING
East            511
South           476
West            359
North           348
South-East      179
North-East      140
South-West      110
North-West       70
EAST             25
SOUTH            20
NORTH            19
North East       11
South East       10
WEST             10
South West        8
SOUTH-WEST        8
EAST-NORTH        8
NORTH-WEST        3
SOUTH-EAST

In [955]:


def clean_facing(value):

    if pd.isna(value):
        return np.nan

    # Convert to string and standardize
    value = str(value).strip().upper()

    # Replace different separators with "-"
    value = value.replace("/", "-")
    value = value.replace(" - ", "-")
    value = value.replace(" -", "-")
    value = value.replace("- ", "-")
    value = value.replace(" ", "-")

    # Standardize direction order
    replacements = {
        "EAST-NORTH": "NORTH-EAST",
        "NORTH-EAST": "NORTH-EAST",

        "EAST-SOUTH": "SOUTH-EAST",
        "SOUTH-EAST": "SOUTH-EAST",

        "WEST-SOUTH": "SOUTH-WEST",
        "SOUTH-WEST": "SOUTH-WEST",

        "WEST-NORTH": "NORTH-WEST",
        "NORTH-WEST": "NORTH-WEST",

        "EAST": "EAST",
        "WEST": "WEST",
        "NORTH": "NORTH",
        "SOUTH": "SOUTH"
    }

    return replacements.get(value, value)

df_clean["FACING"] = df_clean["FACING"].apply(clean_facing)

In [956]:
print(df_clean["FACING"].value_counts(dropna=False))

FACING
EAST          536
SOUTH         498
WEST          371
NORTH         367
SOUTH-EAST    201
NORTH-EAST    164
SOUTH-WEST    127
NORTH-WEST     76
NaN            52
Name: count, dtype: int64


In [957]:
df_clean["FACING"] = df_clean["FACING"].fillna("UNKNOWN")

In [958]:
print(df_clean["FACING"].value_counts())
print(df_clean["FACING"].isnull().sum())

FACING
EAST          536
SOUTH         498
WEST          371
NORTH         367
SOUTH-EAST    201
NORTH-EAST    164
SOUTH-WEST    127
NORTH-WEST     76
UNKNOWN        52
Name: count, dtype: int64
0
